# Ngày 9 · Ngưỡng, ROC/PR và mất cân bằng lớp

- Tính confusion matrix, precision, recall, F1 từ dự đoán
- Dựng đường ROC đúng cả khi có điểm số trùng nhau và tính AUC
- Chọn ngưỡng lớn nhất còn đạt recall yêu cầu

**Đọc:**
- ISLR, §4.4.3 (confusion matrix, ROC trên dữ liệu Default) và §4.5
- Vũ Hữu Tiệp, Machine Learning cơ bản, §5.1 Nhiệm vụ, kinh nghiệm, phép đánh giá

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))
# Breast Cancer Wisconsin (Diagnostic), UCI, có sẵn trong scikit-learn. 569 mẫu, 30 đặc trưng.
_bc = load_breast_cancer()
X_bc = _bc.data
y_bc = 1 - _bc.target   # 1 = ác tính (lớp cần phát hiện), 0 = lành tính
feature_names = list(_bc.feature_names)

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))

# Một phép chia dùng chung cho cả tuần: 70% train, 30% validation, phân tầng theo nhãn.
idx_tr, idx_va = train_test_split(np.arange(len(y_bc)), test_size=0.3, random_state=0, stratify=y_bc)
_scaler = StandardScaler().fit(X_bc[idx_tr])
X_tr, X_va = _scaler.transform(X_bc[idx_tr]), _scaler.transform(X_bc[idx_va])
y_tr, y_va = y_bc[idx_tr], y_bc[idx_va]
from sklearn import metrics as _m
_clf9 = LogisticRegression(max_iter=5000).fit(X_tr, y_tr)
score_va = _clf9.predict_proba(X_va)[:, 1]
_r9 = np.random.default_rng(9)
_yt = _r9.integers(0, 2, 80); _yt[:2] = [0, 1]
_st = np.round(_r9.random(80), 1)            # làm tròn để có nhiều điểm trùng
def _prf_ok(fn):
    for y, yhat in [(_yt, (_st >= .5).astype(int)), (_yt, np.zeros_like(_yt)), (y_va, (score_va >= .3).astype(int))]:
        ref = (_m.precision_score(y, yhat, zero_division=0), _m.recall_score(y, yhat, zero_division=0), _m.f1_score(y, yhat, zero_division=0))
        if not np.allclose(fn(y, yhat), ref):
            return False
    return True
def _brute_threshold(y, score, target):
    for t in np.sort(np.unique(score))[::-1]:
        if (score[y == 1] >= t).mean() >= target:
            return t


## Confusion matrix, precision/recall và ROC-AUC

Gợi ý: Sắp điểm giảm dần; các điểm bằng nhau phải vào cùng một bước của đường ROC. AUC là tổng diện tích các hình thang.

In [ ]:
def confusion(y, yhat):
    # Trả mảng 2×2 [[TN, FP], [FN, TP]].
    raise NotImplementedError

def precision_recall_f1(y, yhat):
    # Trả (precision, recall, f1); khi mẫu số bằng 0 thì đại lượng đó bằng 0.
    raise NotImplementedError

def roc_points(y, score):
    # Trả (fpr, tpr), bắt đầu tại (0, 0), mỗi ngưỡng phân biệt là một điểm.
    raise NotImplementedError

def auc_trapz(fpr, tpr):
    raise NotImplementedError


In [ ]:
_report('d9-metrics',
    _check('Confusion matrix khớp scikit-learn', lambda: (np.array_equal(confusion(y_va, score_va >= .5), _m.confusion_matrix(y_va, score_va >= .5, labels=[0, 1])), 'Thứ tự [[TN, FP], [FN, TP]].')),
    _check('Precision, recall, F1 (kể cả khi không có dự đoán dương)', lambda: (_prf_ok(precision_recall_f1), 'Ba trường hợp, zero_division = 0.')),
    _check('AUC đúng khi có điểm trùng', lambda: (np.isclose(auc_trapz(*roc_points(_yt, _st)), _m.roc_auc_score(_yt, _st)), 'Điểm số làm tròn 1 chữ số thập phân.')),
    _check('Đường ROC bắt đầu (0,0), kết thúc (1,1), không giảm', lambda: ((lambda f, t: f[0] == t[0] == 0 and np.isclose(f[-1], 1) and np.isclose(t[-1], 1) and np.all(np.diff(f) >= 0) and np.all(np.diff(t) >= 0))(*roc_points(y_va, score_va)), 'Trên điểm số validation của logistic.')),
)


## Chọn ngưỡng theo recall yêu cầu

Gợi ý: Duyệt các giá trị điểm phân biệt từ lớn đến nhỏ; dự đoán dương khi score ≥ t. Dừng ở ngưỡng đầu tiên đạt recall mục tiêu.

In [ ]:
def threshold_for_recall(y, score, target):
    # Trả ngưỡng t lớn nhất (trong các giá trị score) sao cho recall của (score >= t) >= target.
    raise NotImplementedError


In [ ]:
_report('d9-threshold',
    _check('Recall ≥ 0,95 với ngưỡng lớn nhất', lambda: (np.isclose(threshold_for_recall(y_va, score_va, .95), _brute_threshold(y_va, score_va, .95)), 'Đối chiếu duyệt toàn bộ ngưỡng.')),
    _check('Đúng với dữ liệu có điểm trùng', lambda: (np.isclose(threshold_for_recall(_yt, _st, .8), _brute_threshold(_yt, _st, .8)), 'Dự đoán dương khi score ≥ t.')),
)


## Dự án

Tải Credit Card Fraud (python py30/download_data.py fraud). Tính tỉ lệ giao dịch gian lận. Huấn luyện logistic trên một phép chia phân tầng, so ROC-AUC với average precision, rồi chọn ngưỡng cho recall ≥ 0,9 và ghi precision tương ứng. Viết hai câu giải thích vì sao accuracy gần như vô nghĩa ở đây.